# manejo de hiperparametros y validacion cruzada

In [ ]:
## GridSearchCV : random forest regresor
# IMPORTACIONES
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# MODELO BASE
model = RandomForestRegressor(
    random_state=42
)

# PIPELINE
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# GRILLA DE HIPERPARÁMETROS
param_grid = {
    "model__n_estimators": [50, 100, 150],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# GRIDSEARCHCV
grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    verbose=2,
    n_jobs=-1
)

# ENTRENAMIENTO
grid_search.fit(X_train, y_train)

# MEJORES PARÁMETROS
print("Mejores parámetros encontrados:", grid_search.best_params_)
print("Mejor R2 score en validación cruzada:", grid_search.best_score_)

# MEJOR MODELO
best_model = grid_search.best_estimator_

# PREDICCIONES
best_preds = best_model.predict(X_test)

# EVALUACIÓN
mae_best = mean_absolute_error(y_test, best_preds)
r2_best = r2_score(y_test, best_preds)

print("MAE del mejor modelo (en X_test):", mae_best)
print("R2 del mejor modelo (en X_test):", r2_best)

In [ ]:
# Gridcearch del segundo modelo de regresion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# QUITAR DATA LEAKAGE
X = df.drop([
    "Global_Sales",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales"
], axis=1)

y = df["Global_Sales"]

# TRAIN TEST
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# columnas categóricas
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

# PREPROCESSOR
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], remainder="passthrough")

# PIPELINE
model = LinearRegression()

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# GRID SEARCH
param_grid = {
    "model__fit_intercept": [True, False],
    "model__positive": [True, False]
}

grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# RESULTADOS
best_model = grid_search.best_estimator_
preds = best_model.predict(X_test)

print("Best params:", grid_search.best_params_)
print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

In [ ]:
# Tercer gridsearch del gradient bosting regresor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# VARIABLES
X = df.drop(columns=['Global_Sales'])
y = df['Global_Sales']

# Detectar columnas categóricas y numéricas
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# PREPROCESAMIENTO
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# TRAIN / TEST
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# MODELO
gbr = GradientBoostingRegressor(random_state=42)

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', gbr)
])

# GRID SEARCH
param_grid = {
    'model__n_estimators': [100, 150],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 5],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# ENTRENAMIENTO
grid_search.fit(X_train, y_train)

# PREDICCIÓN
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# MÉTRICAS
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# RESULTADOS
print("Mejores parámetros encontrados:", grid_search.best_params_)
print("Mejor R2 score en validación cruzada:", grid_search.best_score_)
print("MAE del mejor modelo (X_test):", mae)
print("R2 del mejor modelo (X_test):", r2)

# Gridsearch del Logistic Regression (CLASIFICACION)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# VARIABLES
X = df.drop(columns=['Exitoso'])
y = df['Exitoso']

# Detectar columnas
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# PREPROCESAMIENTO
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=3000))
])

# GridSearch
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Entrenamiento
grid_search.fit(X_train, y_train)

# Predicción
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Métricas
print("Mejores parámetros:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Gridcearch del mejor modelo no supervisado DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

# VARIABLES
X = df.drop(columns=['Global_Sales'])

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# PREPROCESAMIENTO
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

X_processed = preprocessor.fit_transform(X)

# BÚSQUEDA DE PARÁMETROS
param_grid = {
    'eps': [0.3, 0.5, 0.7, 1.0],
    'min_samples': [3, 5, 10]
}

best_score = -1
best_params = None
best_labels = None

for params in ParameterGrid(param_grid):
    dbscan = DBSCAN(
        eps=params['eps'],
        min_samples=params['min_samples']
    )

    labels = dbscan.fit_predict(X_processed)

    # Ignorar casos donde haya solo 1 cluster
    if len(set(labels)) > 1 and -1 in labels:
        score = silhouette_score(X_processed, labels)

        if score > best_score:
            best_score = score
            best_params = params
            best_labels = labels

# RESULTADOS
print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print("Número de clusters:", len(set(best_labels)) - (1 if -1 in best_labels else 0))
print("Ruido (-1):", list(best_labels).count(-1))